# Week 6 – Data Quality Checks (Spark SQL)

This notebook is built specifically for your existing Week 4 Bronze and Week 5 Silver tables.

**Catalog/Schema:** `data_engineering.default`

Silver tables checked:
- `silver_customers`
- `silver_customer_accounts`
- `silver_devices`
- `silver_fraud_cases`
- `silver_merchants`
- `silver_transactions`

The Silver tables are treated as the Week 6 input/candidate layer. Records that pass all checks go to `trusted_*` tables; failed records go to `quarantine_*` tables.


In [0]:
%sql
USE CATALOG data_engineering;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;


active_catalog,active_schema
data_engineering,default


## 1. Input Row Counts


In [0]:
%sql
SELECT 'silver_customers' AS table_name, COUNT(*) AS row_count FROM data_engineering.default.silver_customers
UNION ALL SELECT 'silver_customer_accounts', COUNT(*) FROM data_engineering.default.silver_customer_accounts
UNION ALL SELECT 'silver_devices', COUNT(*) FROM data_engineering.default.silver_devices
UNION ALL SELECT 'silver_fraud_cases', COUNT(*) FROM data_engineering.default.silver_fraud_cases
UNION ALL SELECT 'silver_merchants', COUNT(*) FROM data_engineering.default.silver_merchants
UNION ALL SELECT 'silver_transactions', COUNT(*) FROM data_engineering.default.silver_transactions;


table_name,row_count
silver_customers,18000
silver_customer_accounts,22000
silver_devices,30000
silver_fraud_cases,4200
silver_merchants,2800
silver_transactions,279720


# 2. Customers – Data Quality Checks


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.dq_customers_results
USING DELTA AS
WITH checks AS (
    SELECT
        c.*,
        CASE WHEN customer_id IS NULL OR TRIM(customer_id) = '' THEN 'FAIL' ELSE 'PASS' END AS dq_key,
        CASE WHEN customer_id IN (
            SELECT customer_id
            FROM data_engineering.default.silver_customers
            GROUP BY customer_id
            HAVING COUNT(*) > 1
        ) THEN 'FAIL' ELSE 'PASS' END AS dq_duplicate,
        CASE WHEN source_file_name IS NULL OR ingestion_timestamp IS NULL OR ingestion_run_id IS NULL
             THEN 'FAIL' ELSE 'PASS' END AS dq_lineage
    FROM data_engineering.default.silver_customers c
),
results AS (
    SELECT *,
        CONCAT_WS(',',
            CASE WHEN dq_key = 'FAIL' THEN 'C001_NULL_KEY' END,
            CASE WHEN dq_duplicate = 'FAIL' THEN 'C002_DUPLICATE_KEY' END,
            CASE WHEN dq_lineage = 'FAIL' THEN 'C003_LINEAGE' END
        ) AS failed_rule_ids
    FROM checks
)
SELECT *,
       CASE WHEN failed_rule_ids = '' THEN 'PASS' ELSE 'FAIL' END AS dq_status,
       current_timestamp() AS dq_checked_at
FROM results;


num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.trusted_customers
USING DELTA AS
SELECT * FROM data_engineering.default.dq_customers_results WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE data_engineering.default.quarantine_customers
USING DELTA AS
SELECT * FROM data_engineering.default.dq_customers_results WHERE dq_status = 'FAIL';


num_affected_rows,num_inserted_rows


# 3. Customer Accounts – Null, Duplicate, Reference and Range Checks


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.dq_customer_accounts_results
USING DELTA AS
WITH checks AS (
    SELECT
        a.*,
        CASE WHEN account_id IS NULL OR TRIM(account_id) = '' THEN 'FAIL' ELSE 'PASS' END AS dq_key,
        CASE WHEN account_id IN (
            SELECT account_id
            FROM data_engineering.default.silver_customer_accounts
            GROUP BY account_id
            HAVING COUNT(*) > 1
        ) THEN 'FAIL' ELSE 'PASS' END AS dq_duplicate,
        CASE WHEN c.customer_id IS NULL THEN 'FAIL' ELSE 'PASS' END AS dq_customer_reference,
        CASE WHEN daily_limit_reporting IS NOT NULL AND daily_limit_reporting < 0
             THEN 'FAIL' ELSE 'PASS' END AS dq_range
    FROM data_engineering.default.silver_customer_accounts a
    LEFT JOIN data_engineering.default.silver_customers c
        ON a.customer_id = c.customer_id
),
results AS (
    SELECT *,
        CONCAT_WS(',',
            CASE WHEN dq_key = 'FAIL' THEN 'A001_NULL_KEY' END,
            CASE WHEN dq_duplicate = 'FAIL' THEN 'A002_DUPLICATE_KEY' END,
            CASE WHEN dq_customer_reference = 'FAIL' THEN 'A003_INVALID_CUSTOMER' END,
            CASE WHEN dq_range = 'FAIL' THEN 'A004_NEGATIVE_LIMIT' END
        ) AS failed_rule_ids
    FROM checks
)
SELECT *,
       CASE WHEN failed_rule_ids = '' THEN 'PASS' ELSE 'FAIL' END AS dq_status,
       current_timestamp() AS dq_checked_at
FROM results;


num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.trusted_customer_accounts
USING DELTA AS
SELECT * FROM data_engineering.default.dq_customer_accounts_results WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE data_engineering.default.quarantine_customer_accounts
USING DELTA AS
SELECT * FROM data_engineering.default.dq_customer_accounts_results WHERE dq_status = 'FAIL';


num_affected_rows,num_inserted_rows


# 4. Devices – Key, Duplicate, Customer Reference and Timestamp Checks


In [0]:
%sql

CREATE OR REPLACE TABLE data_engineering.default.dq_devices_results
USING DELTA AS

WITH checks AS (
    SELECT
        d.*,

        CASE 
            WHEN d.device_id IS NULL OR TRIM(d.device_id) = '' 
            THEN 'FAIL' 
            ELSE 'PASS' 
        END AS dq_key,

        CASE 
            WHEN d.device_id IN (
                SELECT device_id
                FROM data_engineering.default.silver_devices
                GROUP BY device_id
                HAVING COUNT(*) > 1
            )
            THEN 'FAIL' 
            ELSE 'PASS' 
        END AS dq_duplicate,

        CASE 
            WHEN c.customer_id IS NULL 
            THEN 'FAIL' 
            ELSE 'PASS' 
        END AS dq_customer_reference,

        CASE 
            WHEN d.last_seen_timestamp < d.first_seen_timestamp
            THEN 'FAIL' 
            ELSE 'PASS' 
        END AS dq_timestamp_order,

        CASE 
            WHEN d.source_file_name IS NULL 
              OR d.ingestion_timestamp IS NULL 
              OR d.ingestion_run_id IS NULL
            THEN 'FAIL' 
            ELSE 'PASS' 
        END AS dq_lineage

    FROM data_engineering.default.silver_devices d

    LEFT JOIN data_engineering.default.silver_customers c
        ON d.customer_id = c.customer_id
),

results AS (
    SELECT *,
        CONCAT_WS(',',
            CASE WHEN dq_key = 'FAIL' THEN 'D001_NULL_KEY' END,
            CASE WHEN dq_duplicate = 'FAIL' THEN 'D002_DUPLICATE_KEY' END,
            CASE WHEN dq_customer_reference = 'FAIL' THEN 'D003_INVALID_CUSTOMER' END,
            CASE WHEN dq_timestamp_order = 'FAIL' THEN 'D004_INVALID_TIMESTAMP_ORDER' END,
            CASE WHEN dq_lineage = 'FAIL' THEN 'D005_LINEAGE' END
        ) AS failed_rule_ids
    FROM checks
)

SELECT *,
       CASE 
           WHEN failed_rule_ids = '' THEN 'PASS' 
           ELSE 'FAIL' 
       END AS dq_status,
       current_timestamp() AS dq_checked_at

FROM results;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.trusted_devices
USING DELTA AS
SELECT * FROM data_engineering.default.dq_devices_results WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE data_engineering.default.quarantine_devices
USING DELTA AS
SELECT * FROM data_engineering.default.dq_devices_results WHERE dq_status = 'FAIL';


num_affected_rows,num_inserted_rows


# 5. Merchants – Key, Duplicate, Date and Lineage Checks


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.dq_merchants_results
USING DELTA AS
WITH checks AS (
    SELECT
        m.*,
        CASE WHEN merchant_id IS NULL OR TRIM(merchant_id) = '' THEN 'FAIL' ELSE 'PASS' END AS dq_key,
        CASE WHEN merchant_id IN (
            SELECT merchant_id FROM data_engineering.default.silver_merchants
            GROUP BY merchant_id HAVING COUNT(*) > 1
        ) THEN 'FAIL' ELSE 'PASS' END AS dq_duplicate,
        CASE WHEN onboarding_date IS NOT NULL AND onboarding_date > CURRENT_DATE()
             THEN 'FAIL' ELSE 'PASS' END AS dq_date_range,
        CASE WHEN source_file_name IS NULL OR ingestion_timestamp IS NULL OR ingestion_run_id IS NULL
             THEN 'FAIL' ELSE 'PASS' END AS dq_lineage
    FROM data_engineering.default.silver_merchants m
),
results AS (
    SELECT *,
        CONCAT_WS(',',
            CASE WHEN dq_key = 'FAIL' THEN 'M001_NULL_KEY' END,
            CASE WHEN dq_duplicate = 'FAIL' THEN 'M002_DUPLICATE_KEY' END,
            CASE WHEN dq_date_range = 'FAIL' THEN 'M003_FUTURE_ONBOARDING_DATE' END,
            CASE WHEN dq_lineage = 'FAIL' THEN 'M004_LINEAGE' END
        ) AS failed_rule_ids
    FROM checks
)
SELECT *, CASE WHEN failed_rule_ids = '' THEN 'PASS' ELSE 'FAIL' END AS dq_status,
       current_timestamp() AS dq_checked_at
FROM results;


num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.trusted_merchants
USING DELTA AS
SELECT * FROM data_engineering.default.dq_merchants_results WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE data_engineering.default.quarantine_merchants
USING DELTA AS
SELECT * FROM data_engineering.default.dq_merchants_results WHERE dq_status = 'FAIL';


num_affected_rows,num_inserted_rows


# 6. Fraud Cases – Key, Duplicate, Transaction Reference and Chronology Checks


In [0]:
%sql

CREATE OR REPLACE TABLE data_engineering.default.dq_fraud_cases_results
USING DELTA AS

WITH checks AS (
    SELECT
        f.*,

        CASE
            WHEN f.case_id IS NULL OR TRIM(f.case_id) = ''
            THEN 'FAIL'
            ELSE 'PASS'
        END AS dq_key,

        CASE
            WHEN f.case_id IN (
                SELECT f2.case_id
                FROM data_engineering.default.silver_fraud_cases f2
                GROUP BY f2.case_id
                HAVING COUNT(*) > 1
            )
            THEN 'FAIL'
            ELSE 'PASS'
        END AS dq_duplicate,

        CASE
            WHEN f.primary_transaction_id IS NOT NULL
                 AND t.transaction_id IS NULL
            THEN 'FAIL'
            ELSE 'PASS'
        END AS dq_transaction_reference,

        CASE
            WHEN f.review_started_timestamp IS NOT NULL
                 AND f.case_created_timestamp IS NOT NULL
                 AND f.review_started_timestamp < f.case_created_timestamp
            THEN 'FAIL'

            WHEN f.case_closed_timestamp IS NOT NULL
                 AND f.review_started_timestamp IS NOT NULL
                 AND f.case_closed_timestamp < f.review_started_timestamp
            THEN 'FAIL'

            ELSE 'PASS'
        END AS dq_chronology,

        CASE
            WHEN f.source_file_name IS NULL
              OR f.ingestion_timestamp IS NULL
              OR f.ingestion_run_id IS NULL
            THEN 'FAIL'
            ELSE 'PASS'
        END AS dq_lineage

    FROM data_engineering.default.silver_fraud_cases f

    LEFT JOIN data_engineering.default.silver_transactions t
        ON f.primary_transaction_id = t.transaction_id
),

results AS (
    SELECT *,
        CONCAT_WS(',',
            CASE WHEN dq_key = 'FAIL' THEN 'F001_NULL_KEY' END,
            CASE WHEN dq_duplicate = 'FAIL' THEN 'F002_DUPLICATE_KEY' END,
            CASE WHEN dq_transaction_reference = 'FAIL' THEN 'F003_INVALID_TRANSACTION' END,
            CASE WHEN dq_chronology = 'FAIL' THEN 'F004_INVALID_CHRONOLOGY' END,
            CASE WHEN dq_lineage = 'FAIL' THEN 'F005_LINEAGE' END
        ) AS failed_rule_ids
    FROM checks
)

SELECT *,
       CASE
           WHEN failed_rule_ids = '' THEN 'PASS'
           ELSE 'FAIL'
       END AS dq_status,
       current_timestamp() AS dq_checked_at

FROM results;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.trusted_fraud_cases
USING DELTA AS
SELECT * FROM data_engineering.default.dq_fraud_cases_results WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE data_engineering.default.quarantine_fraud_cases
USING DELTA AS
SELECT * FROM data_engineering.default.dq_fraud_cases_results WHERE dq_status = 'FAIL';


num_affected_rows,num_inserted_rows


# 7. Transactions – Complete Data Quality Checks

Checks include:
- Null business key
- Duplicate transaction ID
- Customer, account, merchant and device references
- Negative numeric values
- Risk score range
- Timestamp chronology
- Bronze lineage columns


In [0]:
%sql

CREATE OR REPLACE TABLE data_engineering.default.dq_transactions_results
USING DELTA AS

WITH checks AS (
    SELECT
        t.*,

        -- T001: Transaction ID should not be null
        CASE
            WHEN t.transaction_id IS NULL
              OR TRIM(t.transaction_id) = ''
            THEN 'FAIL'
            ELSE 'PASS'
        END AS dq_key,

        -- T002: Duplicate Transaction ID
        CASE
            WHEN t.transaction_id IN (
                SELECT t2.transaction_id
                FROM data_engineering.default.silver_transactions t2
                GROUP BY t2.transaction_id
                HAVING COUNT(*) > 1
            )
            THEN 'FAIL'
            ELSE 'PASS'
        END AS dq_duplicate,

        -- T003: Customer reference
        CASE
            WHEN t.customer_id IS NULL
              OR c.customer_id IS NULL
            THEN 'FAIL'
            ELSE 'PASS'
        END AS dq_customer_reference,

        -- T004: Account reference
        CASE
            WHEN t.account_id IS NULL
              OR a.account_id IS NULL
            THEN 'FAIL'
            ELSE 'PASS'
        END AS dq_account_reference,

        -- T005: Merchant reference
        CASE
            WHEN t.merchant_id IS NULL
              OR m.merchant_id IS NULL
            THEN 'FAIL'
            ELSE 'PASS'
        END AS dq_merchant_reference,

        -- T006: Device reference
        CASE
            WHEN t.device_id IS NULL
              OR d.device_id IS NULL
            THEN 'FAIL'
            ELSE 'PASS'
        END AS dq_device_reference,

        -- T007: Numeric range checks
        CASE
            WHEN t.amount_original IS NOT NULL
                 AND t.amount_original < 0
            THEN 'FAIL'

            WHEN t.amount_reporting_inr IS NOT NULL
                 AND t.amount_reporting_inr < 0
            THEN 'FAIL'

            WHEN t.fx_rate_to_inr IS NOT NULL
                 AND t.fx_rate_to_inr <= 0
            THEN 'FAIL'

            WHEN t.velocity_10m_count IS NOT NULL
                 AND t.velocity_10m_count < 0
            THEN 'FAIL'

            ELSE 'PASS'
        END AS dq_numeric_range,

        -- T008: Risk score should be between 0 and 100
        CASE
            WHEN t.risk_score IS NOT NULL
                 AND (t.risk_score < 0 OR t.risk_score > 100)
            THEN 'FAIL'
            ELSE 'PASS'
        END AS dq_risk_range,

        -- T009: Timestamp chronology
        CASE
            WHEN t.authorization_timestamp IS NOT NULL
                 AND t.event_timestamp IS NOT NULL
                 AND t.authorization_timestamp < t.event_timestamp
            THEN 'FAIL'

            WHEN t.settled_timestamp IS NOT NULL
                 AND t.authorization_timestamp IS NOT NULL
                 AND t.settled_timestamp < t.authorization_timestamp
            THEN 'FAIL'

            WHEN t.outcome_timestamp IS NOT NULL
                 AND t.event_timestamp IS NOT NULL
                 AND t.outcome_timestamp < t.event_timestamp
            THEN 'FAIL'

            ELSE 'PASS'
        END AS dq_chronology,

        -- T010: Lineage columns
        CASE
            WHEN t.source_file_name IS NULL
              OR t.ingestion_timestamp IS NULL
              OR t.ingestion_run_id IS NULL
            THEN 'FAIL'
            ELSE 'PASS'
        END AS dq_lineage

    FROM data_engineering.default.silver_transactions t

    LEFT JOIN data_engineering.default.silver_customers c
        ON t.customer_id = c.customer_id

    LEFT JOIN data_engineering.default.silver_customer_accounts a
        ON t.account_id = a.account_id

    LEFT JOIN data_engineering.default.silver_merchants m
        ON t.merchant_id = m.merchant_id

    LEFT JOIN data_engineering.default.silver_devices d
        ON t.device_id = d.device_id
),

results AS (
    SELECT *,

        CONCAT_WS(',',

            CASE
                WHEN dq_key = 'FAIL'
                THEN 'T001_NULL_KEY'
            END,

            CASE
                WHEN dq_duplicate = 'FAIL'
                THEN 'T002_DUPLICATE_KEY'
            END,

            CASE
                WHEN dq_customer_reference = 'FAIL'
                THEN 'T003_INVALID_CUSTOMER'
            END,

            CASE
                WHEN dq_account_reference = 'FAIL'
                THEN 'T004_INVALID_ACCOUNT'
            END,

            CASE
                WHEN dq_merchant_reference = 'FAIL'
                THEN 'T005_INVALID_MERCHANT'
            END,

            CASE
                WHEN dq_device_reference = 'FAIL'
                THEN 'T006_INVALID_DEVICE'
            END,

            CASE
                WHEN dq_numeric_range = 'FAIL'
                THEN 'T007_INVALID_NUMERIC_RANGE'
            END,

            CASE
                WHEN dq_risk_range = 'FAIL'
                THEN 'T008_INVALID_RISK_SCORE'
            END,

            CASE
                WHEN dq_chronology = 'FAIL'
                THEN 'T009_INVALID_CHRONOLOGY'
            END,

            CASE
                WHEN dq_lineage = 'FAIL'
                THEN 'T010_LINEAGE'
            END

        ) AS failed_rule_ids

    FROM checks
)

SELECT *,

       CASE
           WHEN failed_rule_ids = ''
           THEN 'PASS'
           ELSE 'FAIL'
       END AS dq_status,

       current_timestamp() AS dq_checked_at

FROM results;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.trusted_transactions
USING DELTA AS
SELECT * FROM data_engineering.default.dq_transactions_results WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE data_engineering.default.quarantine_transactions
USING DELTA AS
SELECT * FROM data_engineering.default.dq_transactions_results WHERE dq_status = 'FAIL';


num_affected_rows,num_inserted_rows


# 8. DQ Summary – Required Evidence


In [0]:
%sql
SELECT 'customers' AS entity,
       (SELECT COUNT(*) FROM data_engineering.default.silver_customers) AS candidate_rows,
       (SELECT COUNT(*) FROM data_engineering.default.trusted_customers) AS trusted_rows,
       (SELECT COUNT(*) FROM data_engineering.default.quarantine_customers) AS quarantine_rows
UNION ALL
SELECT 'customer_accounts',
       (SELECT COUNT(*) FROM data_engineering.default.silver_customer_accounts),
       (SELECT COUNT(*) FROM data_engineering.default.trusted_customer_accounts),
       (SELECT COUNT(*) FROM data_engineering.default.quarantine_customer_accounts)
UNION ALL
SELECT 'devices',
       (SELECT COUNT(*) FROM data_engineering.default.silver_devices),
       (SELECT COUNT(*) FROM data_engineering.default.trusted_devices),
       (SELECT COUNT(*) FROM data_engineering.default.quarantine_devices)
UNION ALL
SELECT 'fraud_cases',
       (SELECT COUNT(*) FROM data_engineering.default.silver_fraud_cases),
       (SELECT COUNT(*) FROM data_engineering.default.trusted_fraud_cases),
       (SELECT COUNT(*) FROM data_engineering.default.quarantine_fraud_cases)
UNION ALL
SELECT 'merchants',
       (SELECT COUNT(*) FROM data_engineering.default.silver_merchants),
       (SELECT COUNT(*) FROM data_engineering.default.trusted_merchants),
       (SELECT COUNT(*) FROM data_engineering.default.quarantine_merchants)
UNION ALL
SELECT 'transactions',
       (SELECT COUNT(*) FROM data_engineering.default.silver_transactions),
       (SELECT COUNT(*) FROM data_engineering.default.trusted_transactions),
       (SELECT COUNT(*) FROM data_engineering.default.quarantine_transactions);


entity,candidate_rows,trusted_rows,quarantine_rows
customers,18000,18000,0
customer_accounts,22000,21990,10
devices,30000,29980,20
fraud_cases,4200,4117,83
merchants,2800,2800,0
transactions,279720,278000,1720


## 9. Reconciliation Check

For every entity, the expected result is:

`candidate_rows = trusted_rows + quarantine_rows`


In [0]:
%sql
WITH reconciliation AS (
    SELECT 'customers' AS entity,
           (SELECT COUNT(*) FROM data_engineering.default.silver_customers) AS candidate_rows,
           (SELECT COUNT(*) FROM data_engineering.default.trusted_customers) +
           (SELECT COUNT(*) FROM data_engineering.default.quarantine_customers) AS routed_rows
    UNION ALL
    SELECT 'customer_accounts',
           (SELECT COUNT(*) FROM data_engineering.default.silver_customer_accounts),
           (SELECT COUNT(*) FROM data_engineering.default.trusted_customer_accounts) +
           (SELECT COUNT(*) FROM data_engineering.default.quarantine_customer_accounts)
    UNION ALL
    SELECT 'devices',
           (SELECT COUNT(*) FROM data_engineering.default.silver_devices),
           (SELECT COUNT(*) FROM data_engineering.default.trusted_devices) +
           (SELECT COUNT(*) FROM data_engineering.default.quarantine_devices)
    UNION ALL
    SELECT 'fraud_cases',
           (SELECT COUNT(*) FROM data_engineering.default.silver_fraud_cases),
           (SELECT COUNT(*) FROM data_engineering.default.trusted_fraud_cases) +
           (SELECT COUNT(*) FROM data_engineering.default.quarantine_fraud_cases)
    UNION ALL
    SELECT 'merchants',
           (SELECT COUNT(*) FROM data_engineering.default.silver_merchants),
           (SELECT COUNT(*) FROM data_engineering.default.trusted_merchants) +
           (SELECT COUNT(*) FROM data_engineering.default.quarantine_merchants)
    UNION ALL
    SELECT 'transactions',
           (SELECT COUNT(*) FROM data_engineering.default.silver_transactions),
           (SELECT COUNT(*) FROM data_engineering.default.trusted_transactions) +
           (SELECT COUNT(*) FROM data_engineering.default.quarantine_transactions)
)
SELECT *,
       CASE WHEN candidate_rows = routed_rows THEN 'PASS' ELSE 'FAIL' END AS reconciliation_status
FROM reconciliation;


entity,candidate_rows,routed_rows,reconciliation_status
customers,18000,18000,PASS
customer_accounts,22000,22000,PASS
devices,30000,30000,PASS
fraud_cases,4200,4200,PASS
merchants,2800,2800,PASS
transactions,279720,279720,PASS


# Week 6 Complete

Run the notebook from top to bottom.

Useful screenshots for evidence:
1. Input row counts
2. DQ summary
3. Reconciliation check
4. A trusted table query
5. A quarantine table query


In [0]:
%sql
SELECT transaction_id, failed_rule_ids, dq_status, dq_checked_at
FROM data_engineering.default.dq_transactions_results
WHERE dq_status = 'FAIL'
LIMIT 20;


transaction_id,failed_rule_ids,dq_status,dq_checked_at
TXN0000000060,T009_INVALID_CHRONOLOGY,FAIL,2026-09-05T09:02:50.898Z
TXN0000000851,T007_INVALID_NUMERIC_RANGE,FAIL,2026-09-05T09:02:50.898Z
TXN0000001410,T006_INVALID_DEVICE,FAIL,2026-09-05T09:02:50.898Z
TXN0000001455,T004_INVALID_ACCOUNT,FAIL,2026-09-05T09:02:50.898Z
TXN0000001618,T003_INVALID_CUSTOMER,FAIL,2026-09-05T09:02:50.898Z
TXN0000001647,T009_INVALID_CHRONOLOGY,FAIL,2026-09-05T09:02:50.898Z
TXN0000001901,T009_INVALID_CHRONOLOGY,FAIL,2026-09-05T09:02:50.898Z
TXN0000002101,T005_INVALID_MERCHANT,FAIL,2026-09-05T09:02:50.898Z
TXN0000002176,T008_INVALID_RISK_SCORE,FAIL,2026-09-05T09:02:50.898Z
TXN0000002376,T006_INVALID_DEVICE,FAIL,2026-09-05T09:02:50.898Z


In [0]:
%sql
SELECT *
FROM data_engineering.default.trusted_transactions
LIMIT 10;


physical_record_id,transaction_id,customer_id,account_id,merchant_id,device_id,event_timestamp,authorization_timestamp,transaction_status,amount_original,currency,fx_rate_to_inr,amount_reporting_inr,reporting_currency,channel,country_code,merchant_category,risk_score,risk_band,triggered_rule_ids,is_new_device,location_change_flag,velocity_10m_count,settlement_status,settled_timestamp,case_id,final_outcome,outcome_timestamp,record_updated_timestamp,source_file_name,ingestion_timestamp,ingestion_run_id,dq_key,dq_duplicate,dq_customer_reference,dq_account_reference,dq_merchant_reference,dq_device_reference,dq_numeric_range,dq_risk_range,dq_chronology,dq_lineage,failed_rule_ids,dq_status,dq_checked_at
TXR000000001,TXN0000000001,CUS0008657,ACC00018126,MER000061,DEV00007309,2026-04-08T04:16:28.000Z,2026-04-08T04:17:49.000Z,APPROVED,17208.36,INR,1.000000,17208.36,INR,MOBILE_APP,IN,EDUCATION,29,LOW,SYN-R01_NEW_DEVICE,true,false,2,REVERSED,null,null,PENDING,null,2026-04-08T04:17:49.000Z,transactions_databricks_compatible (1).parquet,2026-09-05T08:49:34.070Z,c4321c9d-5e6f-462d-8e8c-89cfcf1cefbb,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,PASS,2026-09-05T09:02:50.898Z
TXR000000002,TXN0000000002,CUS0013979,ACC00001260,MER002417,DEV00024569,2026-02-07T20:12:27.000Z,2026-02-07T20:13:37.000Z,APPROVED,127.55,AED,22.600000,2882.63,INR,MOBILE_APP,AE,EDUCATION,34,MEDIUM,SYN-R03_CROSS_BORDER|SYN-R06_LOCATION_CHANGE,false,true,2,SETTLED,2026-02-07T20:51:28.000Z,null,PENDING,null,2026-02-07T20:13:37.000Z,transactions_databricks_compatible (1).parquet,2026-09-05T08:49:34.070Z,db17db94-a116-41e4-b14b-38655828c61f,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,PASS,2026-09-05T09:02:50.898Z
TXR000000003,TXN0000000003,CUS0000656,ACC00021895,MER000175,DEV00008999,2026-06-09T10:00:31.000Z,2026-06-09T10:01:52.000Z,APPROVED,2987.08,INR,1.000000,2987.08,INR,CONTACTLESS,IN,DINING,17,LOW,SYN-R00_BASELINE,false,false,3,SETTLED,2026-06-10T16:55:53.000Z,null,PENDING,null,2026-06-09T10:01:52.000Z,transactions_databricks_compatible (1).parquet,2026-09-05T08:49:34.070Z,1b2f70f1-55e5-4e35-b25c-ef1a88fa72fc,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,PASS,2026-09-05T09:02:50.898Z
TXR000000004,TXN0000000004,CUS0010706,ACC00006951,MER000068,DEV00000516,2026-05-05T15:33:42.000Z,2026-05-05T15:33:43.000Z,APPROVED,232.81,USD,83.000000,19323.23,INR,CONTACTLESS,US,ENTERTAINMENT,20,LOW,SYN-R03_CROSS_BORDER,false,false,1,SETTLED,2026-05-06T16:47:08.000Z,null,PENDING,null,2026-05-05T15:33:43.000Z,transactions_databricks_compatible (1).parquet,2026-09-05T08:49:34.070Z,944a93ea-f6ca-4dda-a6fa-e6677d933549,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,PASS,2026-09-05T09:02:50.898Z
TXR000000005,TXN0000000005,CUS0013471,ACC00011798,MER001607,DEV00009221,2026-06-02T20:10:23.000Z,2026-06-02T20:11:33.000Z,APPROVED,17.22,GBP,105.000000,1808.10,INR,POS,GB,FUEL,20,LOW,SYN-R03_CROSS_BORDER|SYN-R06_LOCATION_CHANGE,false,true,2,SETTLED,2026-06-04T14:56:30.000Z,null,PENDING,null,2026-06-02T20:11:33.000Z,transactions_databricks_compatible (1).parquet,2026-09-05T08:49:34.070Z,d347ac33-babb-4449-b685-cfed05894b3a,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,PASS,2026-09-05T09:02:50.898Z
TXR000000006,TXN0000000006,CUS0000618,ACC00015448,MER001685,DEV00025323,2026-04-21T11:25:32.000Z,2026-04-21T11:27:27.000Z,APPROVED,79.27,SGD,62.000000,4914.74,INR,CONTACTLESS,SG,MARKETPLACE,36,MEDIUM,SYN-R03_CROSS_BORDER|SYN-R06_LOCATION_CHANGE,false,true,2,SETTLED,2026-04-24T04:52:19.000Z,null,PENDING,null,2026-04-21T11:27:27.000Z,transactions_databricks_compatible (1).parquet,2026-09-05T08:49:34.070Z,ff7d15d4-1e4b-4786-899b-15aabbfdeab1,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,PASS,2026-09-05T09:02:50.898Z
TXR000000007,TXN0000000007,CUS0010604,ACC00008545,MER002099,DEV00019472,2026-05-25T20:58:19.000Z,2026-05-25T20:58:22.000Z,APPROVED,15207.41,INR,1.000000,15207.41,INR,ECOMMERCE,IN,ELECTRONICS,3,LOW,SYN-R00_BASELINE,false,false,1,SETTLED,2026-05-27T02:22:28.000Z,null,PENDIN